# 실습6 · 필터 — PreFilter · PostFilter · AllowList

"분야가 자동차인 지문 중에서 의미가 가까운 것" 처럼 **유사도 + 조건** 검색을 pgvector 로 해 본다.
강의 : 「벡터DB 코퍼스 생성 전략」 · 「PreFilter / PostFilter / AllowList」 · 「필터링 선택가이드」

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 |
| 2 | 연결과 도구 |
| 3 | 테이블을 나눠도 순위는 같다 — 유사도는 절대값 |
| 4 | 분야별 지문 수 = 필터의 선택도 |
| 5 | PostFilter — HNSW 로 찾고 거르면 |
| 6 | PreFilter — 먼저 거르고 전수 비교 |
| 7 | iterative scan — 모자라면 더 찾아 옴 |
| 8 | 부분 인덱스 — 조건별 그래프 |
| 9 | 세 방식의 행 수 · 정확도 · 시간 |

실습1 의 임베딩과 HNSW 인덱스가 있어야 한다.

### 셀 1 · 설정
- `N_Q` : 셀 5·9 에서 쓰는 질문 수 · `PARTIAL_CATEGORY` : 셀 8 에서 부분 인덱스를 만들 분야

In [ ]:
# 셀 1 · 설정
DB_URL = "postgresql://lab:lab@db:5432/lab"
QID    = 1                  # 셀 3 의 질문
N_Q    = 50                 # 셀 5·9 에서 분야마다 던질 질문 수
PARTIAL_CATEGORY = "자동차"  # 셀 8 부분 인덱스

### 셀 2 · 연결과 도구
- `exact(분야, 질문)` : 인덱스를 끄고 전수 비교한 정답 상위 10 = 정확도의 기준

In [ ]:
# 셀 2 · 연결과 도구
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psycopg
from pgvector.psycopg import register_vector

plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False, "figure.dpi": 110})
pd.set_option("display.max_colwidth", 70)

conn = psycopg.connect(DB_URL, autocommit=True)
conn.prepare_threshold = None      # SET 으로 바꾼 설정이 매번 반영되게 (실행 계획 재사용 끔)
register_vector(conn)              # numpy 배열 ↔ vector 타입


def sql(query, params=None):
    """SELECT 결과를 표(DataFrame)로"""
    cur = conn.execute(query, params)
    return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description]) if cur.description else None


QS = [r[0] for r in conn.execute("SELECT id FROM questions WHERE id %% 11 = 0 ORDER BY id LIMIT %s", (N_Q,))]


from psycopg import sql as pgsql


def plan(query, params):
    """%s 자리에 값을 넣은 쿼리의 실행 계획(줄 목록)"""
    text = pgsql.SQL(query.replace("%s", "{}")).format(*map(pgsql.Literal, params)).as_string(conn)
    return [r[0] for r in conn.execute("EXPLAIN (COSTS OFF) " + text)]


def ids(query, params):
    return [r[0] for r in conn.execute(query, params).fetchall()]


def exact(category, qid):
    """분야 안에서 전수 비교한 상위 10 (정답지)"""
    conn.execute("SET enable_indexscan = off")
    r = ids("""SELECT d.id FROM docs d WHERE d.category = %s
               ORDER BY d.emb128 <=> (SELECT emb128 FROM questions WHERE id = %s), d.id LIMIT 10""", (category, qid))
    conn.execute("RESET enable_indexscan")
    return r

### 셀 3 · 테이블을 나눠도 순위는 같다
코사인 유사도는 두 벡터만으로 정해지는 **절대값** — 다른 지문이 몇 개 있든 같다.
그래서 출처별로 따로 검색해 합쳐도, 한 테이블에서 검색한 결과와 같다.

In [ ]:
# 셀 3 · 한 번에 vs 출처별로 따로 검색해 합치기 (전수 비교)
conn.execute("SET enable_indexscan = off")
one = sql("""SELECT d.id, d.source, round((1 - (d.emb128 <=> q.emb128))::numeric, 4) AS 유사도
             FROM docs d, questions q WHERE q.id = %s ORDER BY d.emb128 <=> q.emb128, d.id LIMIT 10""", (QID,))
split = sql("""SELECT * FROM (
                 SELECT s.id, s.source, s.유사도 FROM unnest(ARRAY['wikipedia', 'hankyung', 'acrofan']) src
                 CROSS JOIN LATERAL (SELECT d.id, d.source, round((1 - (d.emb128 <=> q.emb128))::numeric, 4) AS 유사도
                                     FROM docs d, questions q WHERE q.id = %s AND d.source = src
                                     ORDER BY d.emb128 <=> q.emb128, d.id LIMIT 10) s) x
               ORDER BY 유사도 DESC, id LIMIT 10""", (QID,))
conn.execute("RESET enable_indexscan")
print("같은 결과인가 :", one["id"].tolist() == split["id"].tolist())
pd.concat({"한 테이블": one, "출처별 3번 → 합침": split}, axis=1)

### 셀 4 · 분야별 지문 수 = 필터의 선택도
선택도 = 조건을 통과하는 지문의 비율. 셀 5~9 는 분야 필터로 선택도를 바꿔 가며 잰다.

In [ ]:
# 셀 4 · 분야별 지문 수
CAT = sql("""SELECT category AS 분야, count(*) AS 지문수, round(100.0 * count(*) / (SELECT count(*) FROM docs), 2) AS "비율%"
             FROM docs WHERE category IS NOT NULL GROUP BY category ORDER BY 지문수 DESC""")
CATS = CAT.iloc[[0, 2, 5, 10, 16, 22, 28, len(CAT) - 1]]      # 큰 분야부터 작은 분야까지 8개
display(CATS)
print("분야 수", len(CAT), "· 분야 없음(위키)", conn.execute("SELECT count(*) FROM docs WHERE category IS NULL").fetchone()[0])

### 셀 5 · PostFilter — HNSW 로 찾고 거르면
`WHERE category = … ORDER BY emb128 <=> … LIMIT 10` 을 그냥 쓰면 PG 는 HNSW 로 가까운 후보 **ef_search(40)개**를 먼저 뽑고 그중 조건에 맞는 것만 남긴다.
분야가 작을수록 40개 안에 그 분야가 거의 없다 → LIMIT 10 인데 10개가 안 온다.
단, 플래너는 조건을 통과할 행 수를 추정해 **아주 작은 분야면 인덱스를 버리고 전수 비교**를 고른다 → `실행 계획` 열

In [ ]:
# 셀 5 · PostFilter 의 결과 행 수
POST = """SELECT d.id FROM docs d WHERE d.category = %s
          ORDER BY d.emb128 <=> (SELECT emb128 FROM questions WHERE id = %s) LIMIT 10"""
rows = []
for c, n in CATS[["분야", "지문수"]].values:
    got = [ids(POST, (c, qid)) for qid in QS]
    how = "HNSW 인덱스" if any("docs_emb128_hnsw" in l for l in plan(POST, (c, QS[0]))) else "전수(Seq Scan)"
    rows.append({"분야": c, "지문수": n, "실행 계획": how, "평균 행 수": np.mean([len(g) for g in got]),
                 "정확도": np.mean([len(set(g) & set(exact(c, qid))) / min(10, n) for g, qid in zip(got, QS)])})
P5 = pd.DataFrame(rows)
P5.round(2)

### 셀 6 · PreFilter — 먼저 거르고 전수 비교
`MATERIALIZED` CTE 로 분야를 먼저 걸러 놓고, 그 안에서만 전수 비교. 항상 10개 · 정확도 1.
- 걸러진 지문이 많으면 전수 비교 비용이 커진다 → 셀 9 의 시간

In [ ]:
# 셀 6 · PreFilter
PRE = """WITH f AS MATERIALIZED (SELECT id, emb128 FROM docs WHERE category = %s)
         SELECT f.id FROM f ORDER BY f.emb128 <=> (SELECT emb128 FROM questions WHERE id = %s), f.id LIMIT 10"""
print("\n".join(plan(PRE, ("자동차", QID))))
c = CATS["분야"].iloc[-1]
print(f"\n가장 작은 분야 '{c}' :", "PostFilter", len(ids(POST, (c, QS[0]))), "행 / PreFilter", len(ids(PRE, (c, QS[0]))), "행")

### 셀 7 · iterative scan — 모자라면 더 찾아 옴
pgvector 0.8 : `SET hnsw.iterative_scan = relaxed_order` 면 조건을 통과한 행이 LIMIT 에 모자랄 때 그래프를 **이어서** 탐색한다.
인덱스 탐색을 멈추지 않고 조건에 맞는 것만 모은다 → 강의의 AllowList 에 해당.
- `hnsw.max_scan_tuples`(기본 20,000) 까지 훑고 멈춘다

In [ ]:
# 셀 7 · iterative scan
conn.execute("SET hnsw.iterative_scan = relaxed_order")
rows = []
for c, n in CATS[["분야", "지문수"]].values:
    got = [ids(POST, (c, qid)) for qid in QS]
    rows.append({"분야": c, "지문수": n, "평균 행 수": np.mean([len(g) for g in got]),
                 "정확도": np.mean([len(set(g) & set(exact(c, qid))) / min(10, n) for g, qid in zip(got, QS)])})
conn.execute("RESET hnsw.iterative_scan")
P7 = pd.DataFrame(rows)
pd.concat({"PostFilter": P5.set_index(["분야", "지문수"]).drop(columns="실행 계획"), "iterative scan": P7.set_index(["분야", "지문수"])}, axis=1).round(2)

### 셀 8 · 부분 인덱스 — 조건별 그래프
`CREATE INDEX … WHERE category = '자동차'` : 그 분야 지문만으로 HNSW 그래프를 따로 만든다.
조건이 고정돼 있으면(개인별 · 부서별 등) PreFilter 를 인덱스로 할 수 있다. 실험 뒤 지운다.

In [ ]:
# 셀 8 · 부분 인덱스
conn.execute(f"CREATE INDEX IF NOT EXISTS docs_emb128_part ON docs USING hnsw (emb128 vector_cosine_ops) WHERE category = '{PARTIAL_CATEGORY}'")
print("\n".join(plan(POST, (PARTIAL_CATEGORY, QID))))
got = [ids(POST, (PARTIAL_CATEGORY, qid)) for qid in QS]
print(f"\n'{PARTIAL_CATEGORY}' 부분 인덱스 : 평균 {np.mean([len(g) for g in got]):.1f} 행 · 정확도",
      round(np.mean([len(set(g) & set(exact(PARTIAL_CATEGORY, qid))) / 10 for g, qid in zip(got, QS)]), 3))
_ = conn.execute("DROP INDEX docs_emb128_part")

### 셀 9 · 세 방식의 행 수 · 정확도 · 시간
분야마다 질문 N_Q 개. 시간은 질의 1건 평균(ms). 지문 5,309개라 차이가 작지만 경향을 본다.

In [ ]:
# 셀 9 · 비교
def bench(query, c, setting=None):
    if setting:
        conn.execute(setting)
    t = time.perf_counter()
    got = [ids(query, (c, qid)) for qid in QS]
    ms = (time.perf_counter() - t) / len(QS) * 1000
    if setting:
        conn.execute("RESET ALL")
    return got, ms

rows = []
for c, n in CATS[["분야", "지문수"]].values:
    truth = [exact(c, qid) for qid in QS]
    for name, query, setting in [("PostFilter", POST, None), ("PreFilter", PRE, None),
                                 ("iterative", POST, "SET hnsw.iterative_scan = relaxed_order")]:
        got, ms = bench(query, c, setting)
        rows.append({"분야": c, "지문수": n, "방식": name, "행 수": np.mean([len(g) for g in got]),
                     "정확도": np.mean([len(set(g) & set(t)) / min(10, n) for g, t in zip(got, truth)]), "ms": ms})
B9 = pd.DataFrame(rows)
fig, ax = plt.subplots(1, 3, figsize=(16, 3.6))
for name, col in [("PostFilter", "#9aa5a8"), ("PreFilter", "#3D484A"), ("iterative", "#FF7043")]:
    s = B9[B9["방식"] == name]
    for a, m in zip(ax, ["행 수", "정확도", "ms"]):
        a.plot(100 * s["지문수"] / 5309, s[m], "-o", color=col, label=name)
for a, m in zip(ax, ["결과 행 수 (LIMIT 10)", "정확도 (전수 비교 대비)", "질의당 ms"]):
    a.set_xscale("log"); a.set_xlabel("선택도 % (로그)"); a.set_title(m); a.legend()
plt.tight_layout(); plt.show()
B9.pivot_table(index=["분야", "지문수"], columns="방식", values=["행 수", "정확도", "ms"]).sort_index(level=1, ascending=False).round(2)